# Очистка датасета VKR

Этот ноутбук берёт грязные изображения из:

`/content/drive/MyDrive/vkr/images_for_training`

и сохраняет очищенные изображения в:

`/content/drive/MyDrive/vkr/data_clean`

Битые изображения переносятся в:

`/content/drive/MyDrive/vkr/quarantine_invalid`

## 1. Установка зависимостей

Запусти эту ячейку один раз.  
После выполнения обязательно нажми:

**Среда выполнения → Перезапустить среду выполнения**

Потом запускай ноутбук заново со следующей ячейки.

In [ ]:
!pip uninstall -y numpy rembg onnxruntime opencv-python-headless
!pip install --no-cache-dir \
    numpy==1.26.4 \
    onnxruntime==1.18.1 \
    opencv-python-headless==4.10.0.84 \
    rembg==2.0.57 \
    pillow \
    tqdm

print("Готово. Теперь перезапусти среду выполнения и запускай ноутбук дальше.")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 162.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 183.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 162.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 165.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 152.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

Готово. Теперь перезапусти среду выполнения и запускай ноутбук дальше.


## 2. Подключение Google Drive

In [ ]:
from google.colab import drive

# force_remount=True нужен, чтобы не ловить ошибку:
# ValueError: Mountpoint must not already contain files
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 3. Пути к данным

Структура на Google Drive должна быть такой:

```text
MyDrive/
└── vkr/
    ├── images_for_training      # грязные изображения
    ├── data_clean               # сюда сохраняются чистые изображения
    └── quarantine_invalid       # сюда уходят битые файлы
```

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/vkr")

DIRTY_DIR = BASE_DIR / "images_for_training"
CLEAN_DIR = BASE_DIR / "data_clean"
QUARANTINE_DIR = BASE_DIR / "quarantine_invalid"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DIRTY_DIR:", DIRTY_DIR)
print("CLEAN_DIR:", CLEAN_DIR)
print("QUARANTINE_DIR:", QUARANTINE_DIR)

print("\nПроверка папок:")
print("BASE_DIR exists:", BASE_DIR.exists())
print("DIRTY_DIR exists:", DIRTY_DIR.exists())
print("CLEAN_DIR exists:", CLEAN_DIR.exists())
print("QUARANTINE_DIR exists:", QUARANTINE_DIR.exists())

BASE_DIR: /content/drive/MyDrive/vkr
DIRTY_DIR: /content/drive/MyDrive/vkr/images_for_training
CLEAN_DIR: /content/drive/MyDrive/vkr/data_clean
QUARANTINE_DIR: /content/drive/MyDrive/vkr/quarantine_invalid

Проверка папок:
BASE_DIR exists: True
DIRTY_DIR exists: True
CLEAN_DIR exists: True
QUARANTINE_DIR exists: True


## 4. Диагностика входной папки

Эта ячейка показывает, видит ли Colab твои грязные данные.
Если тут `0`, значит проблема не в очистке, а в том, что Colab не видит файлы в `images_for_training`.

In [ ]:
print("Что лежит в BASE_DIR:")
if BASE_DIR.exists():
    for p in BASE_DIR.iterdir():
        print("-", p.name)
else:
    print("BASE_DIR не найден")

print("\nПервые объекты в DIRTY_DIR:")
if DIRTY_DIR.exists():
    all_items = list(DIRTY_DIR.rglob("*"))
    print("Всего объектов:", len(all_items))
    for p in all_items[:30]:
        print(p)
else:
    print("DIRTY_DIR не найден")

Что лежит в BASE_DIR:
- images_for_training
- data_clean
- quarantine_invalid

Первые объекты в DIRTY_DIR:
Всего объектов: 15000
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004091_03_4_full.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004203_04_7_additional.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004460_01_4_full.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004369_03_3_back.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004291_03_1_front.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004404_03_3_back.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004353_03_7_additional.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004351_02_1_front.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id_00004496_05_1_front.jpg
/content/drive/MyDrive/vkr/images_for_training/WOMEN3_Dresses_id

## 5. Импорты и настройки очистки

In [ ]:
import io
import os
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Literal

from PIL import Image, UnidentifiedImageError
from rembg import new_session, remove
from tqdm.auto import tqdm

BackgroundMode = Literal["transparent", "white", "black", "auto"]


@dataclass(frozen=True)
class RemoveBgOptions:
    model: str = "u2net"
    background: BackgroundMode = "auto"
    auto_background_threshold: int = 160
    skip_invalid: bool = True
    skip_solid_background: bool = True
    solid_background_tolerance: int = 18
    solid_background_edge_fraction: float = 0.9


@dataclass(frozen=True)
class ImageValidationResult:
    path: Path
    is_valid: bool
    error: str | None = None

## 6. Вспомогательные функции

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def iter_images(root: Path) -> Iterable[Path]:
    root = Path(root)
    if not root.exists():
        return
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            yield path


def validate_image_file(path: Path) -> ImageValidationResult:
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img.load()
        return ImageValidationResult(path=path, is_valid=True)
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return ImageValidationResult(path=path, is_valid=False, error=str(exc))


def move_to_quarantine(path: Path, source_root: Path, quarantine_dir: Path) -> Path:
    try:
        relative_path = path.relative_to(source_root)
    except ValueError:
        relative_path = Path(path.name)

    destination = quarantine_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(destination))
    return destination


def is_near_color(r: int, g: int, b: int, target: str, tolerance: int) -> bool:
    if target == "white":
        return r >= 255 - tolerance and g >= 255 - tolerance and b >= 255 - tolerance
    return r <= tolerance and g <= tolerance and b <= tolerance


def has_solid_black_or_white_background(
    image: Image.Image,
    tolerance: int,
    edge_fraction: float,
) -> bool:
    rgb = image.convert("RGB")
    width, height = rgb.size

    if width < 3 or height < 3:
        return False

    edge_pixels = []

    for x in range(width):
        edge_pixels.append(rgb.getpixel((x, 0)))
        edge_pixels.append(rgb.getpixel((x, height - 1)))

    for y in range(1, height - 1):
        edge_pixels.append(rgb.getpixel((0, y)))
        edge_pixels.append(rgb.getpixel((width - 1, y)))

    for target in ("white", "black"):
        matches = sum(
            1
            for r, g, b in edge_pixels
            if is_near_color(r, g, b, target, tolerance)
        )
        if matches / len(edge_pixels) >= edge_fraction:
            return True

    return False


def coerce_image(output: object) -> Image.Image:
    if isinstance(output, Image.Image):
        return output
    if isinstance(output, (bytes, bytearray, memoryview)):
        return Image.open(io.BytesIO(bytes(output)))
    raise TypeError(f"Unsupported rembg output type: {type(output)!r}")


def select_contrast_background(result: Image.Image, threshold: int) -> str:
    rgba = result.convert("RGBA")
    weighted_luminance = 0.0
    alpha_total = 0.0

    for r, g, b, a in rgba.getdata():
        if a == 0:
            continue

        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        weighted_luminance += luminance * a
        alpha_total += a

    if alpha_total == 0:
        return "white"

    return "black" if (weighted_luminance / alpha_total) >= threshold else "white"


def apply_background(
    result: Image.Image,
    background: BackgroundMode,
    threshold: int,
) -> Image.Image:
    if background == "transparent":
        return result

    resolved = background
    if background == "auto":
        resolved = select_contrast_background(result, threshold)

    color = (255, 255, 255, 255) if resolved == "white" else (0, 0, 0, 255)
    bg = Image.new("RGBA", result.size, color)

    return Image.alpha_composite(bg, result).convert("RGB")

def save_image_safely(image: Image.Image, output_path: Path) -> None:
    """
    Safe save for PNG/JPG/JPEG.

    JPEG cannot save RGBA images.
    So if output is .jpg/.jpeg, image is converted to RGB first.
    """
    output_path = Path(output_path)
    suffix = output_path.suffix.lower()

    if suffix in {".jpg", ".jpeg"}:
        if image.mode == "RGBA":
            background = Image.new("RGB", image.size, (255, 255, 255))
            background.paste(image, mask=image.split()[-1])
            image = background
        elif image.mode != "RGB":
            image = image.convert("RGB")

    image.save(output_path)


## 7. Проверка датасета перед очисткой

In [ ]:
image_paths = list(iter_images(DIRTY_DIR))

print("Найдено изображений:", len(image_paths))
print("\nПервые 20 изображений:")
for p in image_paths[:20]:
    print(p)

validation_results = [
    validate_image_file(path)
    for path in tqdm(image_paths, desc="Validating images")
]

invalid_results = [
    result
    for result in validation_results
    if not result.is_valid
]

print("\nTotal images:", len(validation_results))
print("Invalid images:", len(invalid_results))

if invalid_results:
    print("\nПервые 10 битых файлов:")
    for result in invalid_results[:10]:
        print(result.path, "::", result.error)

## 8. Основная функция очистки

Логика:

1. Берём изображение из `images_for_training`.
2. Если файл битый — переносим в `quarantine_invalid`.
3. Если фон уже чёрный или белый — просто копируем в `data_clean`.
4. Если фон грязный — прогоняем через `rembg` и сохраняем результат в `data_clean`.

In [ ]:
def process_dataset(
    dirty_dir: Path,
    clean_dir: Path,
    quarantine_dir: Path,
    options: RemoveBgOptions,
):
    dirty_dir = Path(dirty_dir)
    clean_dir = Path(clean_dir)
    quarantine_dir = Path(quarantine_dir)

    clean_dir.mkdir(parents=True, exist_ok=True)
    quarantine_dir.mkdir(parents=True, exist_ok=True)

    image_paths = list(iter_images(dirty_dir))

    print("Dirty dir:", dirty_dir)
    print("Clean dir:", clean_dir)
    print("Quarantine dir:", quarantine_dir)
    print("Images found:", len(image_paths))

    if len(image_paths) == 0:
        print("\nВНИМАНИЕ: изображения не найдены.")
        print("Проверь, что файлы лежат именно здесь:")
        print(dirty_dir)
        return {
            "found": 0,
            "processed": 0,
            "copied_solid_background": 0,
            "skipped_invalid": 0,
        }

    session = new_session(options.model)

    processed = 0
    copied_solid_background = 0
    skipped_invalid = 0

    for image_path in tqdm(image_paths, desc="Cleaning dataset"):
        validation = validate_image_file(image_path)

        if not validation.is_valid:
            move_to_quarantine(image_path, dirty_dir, quarantine_dir)
            skipped_invalid += 1

            if options.skip_invalid:
                continue

            raise ValueError(f"Broken image: {image_path} :: {validation.error}")

        relative_path = image_path.relative_to(dirty_dir)

        if options.background == "transparent":
            output_suffix = ".png"
        else:
            output_suffix = image_path.suffix

        output_path = (clean_dir / relative_path).with_suffix(output_suffix)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with Image.open(image_path) as img:
            if options.skip_solid_background and has_solid_black_or_white_background(
                img,
                tolerance=options.solid_background_tolerance,
                edge_fraction=options.solid_background_edge_fraction,
            ):
                save_image_safely(img.copy(), output_path)
                copied_solid_background += 1
                continue

            rgba = img.convert("RGBA")
            result = coerce_image(remove(rgba, session=session)).convert("RGBA")
            result = apply_background(
                result,
                options.background,
                options.auto_background_threshold,
            )
            save_image_safely(result, output_path)
            processed += 1

    return {
        "found": len(image_paths),
        "processed": processed,
        "copied_solid_background": copied_solid_background,
        "skipped_invalid": skipped_invalid,
    }

## 9. Запуск очистки

In [ ]:
options = RemoveBgOptions(
    model="u2net",
    background="auto",
    auto_background_threshold=160,
    skip_invalid=True,
    skip_solid_background=True,
)

summary = process_dataset(
    dirty_dir=DIRTY_DIR,
    clean_dir=CLEAN_DIR,
    quarantine_dir=QUARANTINE_DIR,
    options=options,
)

summary

## 9.1. Повторный запуск после ошибки

Если очистка упала на середине, запускай эту ячейку. Уже готовые изображения в `data_clean` будут пропущены.


In [ ]:
def process_dataset_resume(
    dirty_dir: Path,
    clean_dir: Path,
    quarantine_dir: Path,
    options: RemoveBgOptions,
):
    dirty_dir = Path(dirty_dir)
    clean_dir = Path(clean_dir)
    quarantine_dir = Path(quarantine_dir)

    clean_dir.mkdir(parents=True, exist_ok=True)
    quarantine_dir.mkdir(parents=True, exist_ok=True)

    image_paths = list(iter_images(dirty_dir))

    print("Dirty dir:", dirty_dir)
    print("Clean dir:", clean_dir)
    print("Quarantine dir:", quarantine_dir)
    print("Images found:", len(image_paths))

    session = new_session(options.model)

    processed = 0
    copied_solid_background = 0
    skipped_invalid = 0
    skipped_existing = 0
    failed = 0

    for image_path in tqdm(image_paths, desc="Cleaning dataset"):
        relative_path = image_path.relative_to(dirty_dir)

        if options.background == "transparent":
            output_suffix = ".png"
        else:
            output_suffix = image_path.suffix.lower()

        output_path = (clean_dir / relative_path).with_suffix(output_suffix)

        if output_path.exists():
            skipped_existing += 1
            continue

        output_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            validation = validate_image_file(image_path)

            if not validation.is_valid:
                move_to_quarantine(image_path, dirty_dir, quarantine_dir)
                skipped_invalid += 1

                if options.skip_invalid:
                    continue

                raise ValueError(f"Broken image: {image_path} :: {validation.error}")

            with Image.open(image_path) as img:
                if options.skip_solid_background and has_solid_black_or_white_background(
                    img,
                    tolerance=options.solid_background_tolerance,
                    edge_fraction=options.solid_background_edge_fraction,
                ):
                    save_image_safely(img.copy(), output_path)
                    copied_solid_background += 1
                    continue

                rgba = img.convert("RGBA")
                result = coerce_image(remove(rgba, session=session)).convert("RGBA")
                result = apply_background(
                    result,
                    options.background,
                    options.auto_background_threshold,
                )
                save_image_safely(result, output_path)
                processed += 1

        except Exception as exc:
            failed += 1
            print(f"FAILED: {image_path} :: {exc}")

    return {
        "found": len(image_paths),
        "processed": processed,
        "copied_solid_background": copied_solid_background,
        "skipped_existing": skipped_existing,
        "skipped_invalid": skipped_invalid,
        "failed": failed,
    }


options = RemoveBgOptions(
    model="u2net",
    background="auto",
    auto_background_threshold=160,
    skip_invalid=True,
    skip_solid_background=True,
)

summary = process_dataset_resume(
    dirty_dir=DIRTY_DIR,
    clean_dir=CLEAN_DIR,
    quarantine_dir=QUARANTINE_DIR,
    options=options,
)

summary

Dirty dir: /content/drive/MyDrive/vkr/images_for_training
Clean dir: /content/drive/MyDrive/vkr/data_clean
Quarantine dir: /content/drive/MyDrive/vkr/quarantine_invalid
Images found: 15000


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

Cleaning dataset:   0%|          | 0/15000 [00:00<?, ?it/s]

{'found': 15000,
 'processed': 9906,
 'copied_solid_background': 4089,
 'skipped_existing': 1005,
 'skipped_invalid': 0,
 'failed': 0}

## 10. Контроль результата

In [ ]:
dirty_count = len(list(iter_images(DIRTY_DIR)))
clean_count = len(list(iter_images(CLEAN_DIR)))
quarantine_count = len(list(QUARANTINE_DIR.rglob("*"))) if QUARANTINE_DIR.exists() else 0

print("Dirty images left:", dirty_count)
print("Clean images:", clean_count)
print("Quarantine objects:", quarantine_count)

print("\nПервые 20 файлов в data_clean:")
for p in list(CLEAN_DIR.rglob("*"))[:20]:
    if p.is_file():
        print(p)

## 11. Просмотр нескольких очищенных изображений

In [ ]:
import matplotlib.pyplot as plt

clean_images = list(iter_images(CLEAN_DIR))

if not clean_images:
    print("В data_clean пока нет изображений.")
else:
    for img_path in clean_images[:5]:
        img = Image.open(img_path)
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_path.name)
        plt.show()